In [2]:
# 1. Монтируем Google Диск (если еще не примонтирован)
from google.colab import drive
drive.mount("/content/drive")

!unzip -q /content/drive/MyDrive/CASIA2.zip -d /content/raw_data/

Mounted at /content/drive


In [3]:
import os
import sys
import shutil
import os
if os.path.exists('/content/src'):
    shutil.rmtree('/content/src')
shutil.copytree('/content/drive/MyDrive/photo_forensics_unet/src', '/content/src')
import pickle
import tensorflow as tf

EarlyStopping = tf.keras.callbacks.EarlyStopping
ModelCheckpoint = tf.keras.callbacks.ModelCheckpoint
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau

PROJECT_ROOT = '/content/drive/MyDrive/photo_forensics_unet'

# Add PROJECT_ROOT to sys.path so Python can find modules like 'utils'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from utils.train_val_split import get_train_val_generators
from src.unet_model import build_unet
from utils.img_processing import load_gt_mask, compute_ela
from utils.build_pairs import build_data_pairs
import numpy as np
from sklearn.model_selection import train_test_split


TP_DIR = "/content/raw_data/Tp"                  # Путь к папке с манипуляциями
GT_DIR = "/content/raw_data/CASIA 2 Groundtruth" # Путь к папке с масками

# Настройки обучения
BATCH_SIZE = 16  # На T4 в Colab 16 батч должен зайти идеально для разрешения 256x256
IMG_SIZE = 256
EPOCHS = 30      # 30 эпох с ранней остановкой хватит за глаза, чтобы сеть поняла геометрию


# 1. Собираем пары путей (убедись, что твои TP_DIR и GT_DIR прописаны выше)
pairs = build_data_pairs(TP_DIR, GT_DIR)
print(f"Найдено пар для обработки: {len(pairs)}")

X_data = []
Y_data = []

print("Начинается препроцессинг ELA в оперативную память...")
for i, pair in enumerate(pairs):
    try:
        # Считаем ELA (массив float от 0.0 до 1.0)
        ela_img = compute_ela(pair['tp_path'], quality=90, scale=15, target_size=(256, 256))

        #ЖЕСТКАЯ НОРМАЛИЗАЦИЯ (чтобы не было черных квадратов)
        if ela_img.max() - ela_img.min() > 0:
            ela_img = (ela_img - ela_img.min()) / (ela_img.max() - ela_img.min())
        else:
            ela_img = np.zeros((256, 256, 3), dtype=np.float32)

        # Загружаем истинную маску фотошопа
        mask = load_gt_mask(pair['gt_path'], target_size=(256, 256))

        X_data.append(ela_img.astype(np.float32))
        Y_data.append(mask.astype(np.float32))

        if (i + 1) % 500 == 0:
            print(f"Обработано {i + 1} из {len(pairs)} картинок")
    except Exception as e:
        continue

# Конвертируем в монолитные numpy-массивы
X_data = np.array(X_data)
Y_data = np.array(Y_data)

print(f"Итоговый размер тензора признаков ELA: {X_data.shape}")
print(f"Итоговый размер тензора масок: {Y_data.shape}")

# Делим на трейн и валидацию (85% на 15%)
X_train, X_val, Y_train, Y_val = train_test_split(X_data, Y_data, test_size=0.15, random_state=42)
print(f"Обучение: {X_train.shape[0]} кадров, Валидация: {X_val.shape[0]} кадров")

print("Инициализация генераторов данных...")
train_gen, val_gen, train_steps, val_steps = get_train_val_generators(
    tp_dir=TP_DIR,
    gt_dir=GT_DIR,
    batch_size=BATCH_SIZE,
    target_size=(IMG_SIZE, IMG_SIZE),
    split_ratio=0.15
)

print("Сборка архитектуры U-Net...")
model = build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3))

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

callbacks = [
    # Следим за честным падением ошибки на валидации
    EarlyStopping(monitor='val_loss', mode='min', patience=12, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "best_unet_forensics.keras"), monitor='val_loss', mode='min', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Старт локального обучения U-Net напрямую из ОЗУ...")
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=16,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)
print("\n🏆 Финиш! Модель идеально обучена и сохранена в папку models.")

# 5. Сохраняем историю обучения для графиков в курсовую
history_path = os.path.join(PROJECT_ROOT, "models", "history_unet.pkl")
with open(history_path, 'wb') as f:
    pickle.dump(history.history, f)

print("\n ОБУЧЕНИЕ ПОЛНОСТЬЮ ЗАВЕРШЕНО!")
print(f"Лучшая модель сохранена в: {PROJECT_ROOT}/models/best_unet_forensics.keras")

Сканирование датасета и сопоставление пар...
Количество успешно сопоставленных пар для сегментации: 2305
Найдено пар для обработки: 2305
Начинается препроцессинг ELA в оперативную память...
Обработано 500 из 2305 картинок
Обработано 1000 из 2305 картинок
Обработано 1500 из 2305 картинок
Обработано 2000 из 2305 картинок
Итоговый размер тензора признаков ELA: (2305, 256, 256, 3)
Итоговый размер тензора масок: (2305, 256, 256, 1)
Обучение: 1959 кадров, Валидация: 346 кадров
Инициализация генераторов данных...
Сканирование датасета и сопоставление пар...
Количество успешно сопоставленных пар для сегментации: 2305
Распределение: 1959 пар на обучение, 346 пар на валидацию.
Сборка архитектуры U-Net...
Старт локального обучения U-Net напрямую из ОЗУ...
Epoch 1/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5236 - dice_coefficient: 0.1789 - dice_loss: 0.8211 - iou_metric: 0.0993 - loss: 1.0155   
Epoch 1: val_loss improved from None to 0.97258, saving model to models/best_unet_forensi

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

if os.path.exists('/content/src'):
    shutil.rmtree('/content/src')
shutil.copytree('/content/drive/MyDrive/photo_forensics_unet/src', '/content/src')
import sys
PROJECT_ROOT = '/content/drive/MyDrive/photo_forensics_unet'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
from utils.build_pairs import build_data_pairs
from utils.img_processing import load_gt_mask, compute_ela
from utils.metrics import iou_metric, dice_coefficient, bce_dice_loss, dice_loss

#Настройка пути
PROJECT_PATH = "/content/drive/MyDrive/photo_forensics_unet"
MODEL_PATH = os.path.join(PROJECT_PATH, "models", "best_unet_forensics.keras")
RESULTS_DIR = os.path.join(PROJECT_PATH, "results_vis")
os.makedirs(RESULTS_DIR, exist_ok=True)

#TP_DIR, GT_DIR - Изображения фотошопа и маски
TP_DIR = "/content/raw_data/Tp"
GT_DIR = "/content/raw_data/CASIA 2 Groundtruth"

#Загружаем сохраненную модель, обязательно передавая кастомные метрики
print("Загрузка обученной U-Net модели...")
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'iou_metric': iou_metric,
        'dice_coefficient': dice_coefficient,
        'bce_dice_loss': bce_dice_loss,
        'dice_loss': dice_loss
    }
)
print("Модель успешно загружена!")

#Собираем пары картинок и выбираем случайные примеры
pairs = build_data_pairs(TP_DIR, GT_DIR)
np.random.seed(42) # Фиксируем сид, чтобы картинки не менялись при перезапусках
shuffled_indices = np.random.permutation(len(pairs))

num_examples = 5
count = 0

print(f"Генерируем {num_examples} тестовых коллажей...")

for idx in shuffled_indices:
    if count >= num_examples:
        break

    pair = pairs[idx]
    try:
        # На лету готовим ELA и Маску для визуализации
        ela_img = compute_ela(pair['tp_path'], target_size=(256, 256))
        true_mask = load_gt_mask(pair['gt_path'], target_size=(256, 256))

        # Модель ждет батч (размерность 1, 256, 256, 3)
        input_tensor = np.expand_dims(ela_img, axis=0)
        print(ela_img.max(), ela_img.min())
        # Делаем предсказание
        pred_raw = model.predict(input_tensor, verbose=0)[0]
        # Бинаризуем по порогу 0.5
        pred_mask = (pred_raw.squeeze() > 0.5).astype(np.uint8)

        # Рисуем красивый график для диплома/курсовой
        plt.figure(figsize=(12, 4))

        # Левый график - оригинальный зашумленный ELA
        plt.subplot(1, 3, 1)
        plt.imshow(ela_img)
        plt.title("Входной ELA-кадр")
        plt.axis('off')

        # Средний график - Истинная маска (Groundtruth из CASIA)
        plt.subplot(1, 3, 2)
        plt.imshow(true_mask.squeeze(), cmap='gray')
        plt.title("Истинная маска (y_true)")
        plt.axis('off')

        # Правый график - То, что нарисовала твоя сетка
        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask, cmap='gray')
        plt.title("Предсказание U-Net (y_pred)")
        plt.axis('off')

        plt.tight_layout()
        save_path = os.path.join(RESULTS_DIR, f"saved_prediction_{count+1}.png")
        plt.savefig(save_path, dpi=150)
        plt.close()

        print(f"Пример {count+1} сохранен в Гугл Диск -> {save_path}")
        count += 1

    except Exception as e:
        # Если какой-то файл не прочитался, просто берем следующий
        continue

print(f"\nГотово! Проверяй папку 'results_vis' на своем Гугл Диске.")